## Agentic AI Code Converter

In [ ]:
import json
import platform
from openai import OpenAI
import subprocess
import time
import json
import pandas as pd
import traceback
import gradio as gr
import pandas as pd
from dotenv import load_dotenv

In [ ]:
load_dotenv()
client = OpenAI()

In [ ]:
PLANNER_SYSTEM_PROMPT = """
You are the Planner Agent in an autonomous code optimization system.

Your ONLY responsibility is to analyze Python code and create an optimization plan.

Do NOT generate Rust or C++ code.

Analyze the Python code and return a JSON object with exactly the following schema:

{
    "summary": "...",

    "algorithm_type": "...",

    "time_complexity": "...",

    "space_complexity": "...",

    "bottlenecks": [
        "...",
        "..."
    ],

    "parallelizable": true,

    "recommended_language": "Rust",

    "optimization_plan": [
        "...",
        "...",
        "..."
    ]
}

Rules:

- Think carefully before answering.
- Be specific.
- Optimization plan should contain concrete implementation steps.
- Return ONLY valid JSON.
"""


def planner_prompt(python_code: str):

    return f"""
    Analyze the following Python code.

    python
    {python_code}
    
    Return ONLY the required JSON.
    """
    
def planner_agent(client, model, python_code):
    response = client.chat.completions.create(
        model=model,
        reasoning_effort="high",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": PLANNER_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": planner_prompt(python_code)
            }
        ]
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
BUILDER_SYSTEM_PROMPT = """
You are the Builder Agent.

Your responsibility is to generate HIGH PERFORMANCE compilable
Rust or C++ code.

Rules:

1. Follow the optimization plan exactly.

2. Produce ONLY source code.

3. No markdown.

4. No explanations.

5. If compiler errors are supplied,
repair ONLY those errors.

6. Never change program behaviour.

7. Preserve identical output.
"""


def builder_prompt(python_code, plan, compile_error=None):

    prompt = f"""
    Python Code

    {python_code}

    Optimization Plan

    {plan}

    Target Language

    {plan["recommended_language"]}

    """

    if compile_error:

        prompt += f"""
        Previous compilation failed.

        Compiler Error

        {compile_error}

        Fix the program.

        Return ONLY corrected source code.
        """

    else:
        prompt += """
        Generate the optimized implementation.

        Return ONLY source code.
        """

    return prompt


def generate_code(client, model, python_code, plan, compile_error=None):

    response = client.chat.completions.create(

        model=model,

        reasoning_effort="high",

        messages=[

            {
                "role": "system",
                "content": BUILDER_SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": builder_prompt(
                    python_code,
                    plan,
                    compile_error
                )
            }

        ]
    )

    code = response.choices[0].message.content

    code = code.replace("```cpp", "")
    code = code.replace("```rust", "")
    code = code.replace("```", "")

    return code.strip()


def write_source(code, extension):

    filename = f"main.{extension}"

    with open(filename, "w") as f:
        f.write(code)

    return filename


def compile_source(compile_command):

    try:

        subprocess.run(
            compile_command,
            check=True,
            capture_output=True,
            text=True
        )

        return True, ""

    except subprocess.CalledProcessError as e:

        return False, e.stderr


def builder_agent(client, model, python_code, plan, compile_command, extension, max_retries=3):

    compile_error = None

    for attempt in range(max_retries):

        print(f"\nBuilder Attempt {attempt+1}")

        code = generate_code(
            client,
            model,
            python_code,
            plan,
            compile_error
        )

        write_source(code, extension)

        success, compile_error = compile_source(
            compile_command
        )

        if success:

            print("Compilation Successful")

            return {

                "success": True,

                "source_code": code,

                "attempts": attempt + 1,

                "compile_error": None

            }

        print("Compilation Failed")

    return {

        "success": False,

        "source_code": code,

        "attempts": max_retries,

        "compile_error": compile_error

    }

In [ ]:
def run_python(python_file):

    start = time.perf_counter()

    result = subprocess.run(
        ["python", python_file],
        capture_output=True,
        text=True
    )

    runtime = time.perf_counter() - start

    return {
        "success": result.returncode == 0,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
        "runtime": runtime
    }


def run_binary(binary_path):

    start = time.perf_counter()

    result = subprocess.run(
        [binary_path],
        capture_output=True,
        text=True
    )

    runtime = time.perf_counter() - start

    return {
        "success": result.returncode == 0,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
        "runtime": runtime
    }


def compare_outputs(python_result, binary_result):

    return python_result["stdout"] == binary_result["stdout"]


def calculate_speedup(python_time, compiled_time):

    if compiled_time <= 0:
        return float("inf")

    return python_time / compiled_time


def create_validation_report(
    python_result,
    binary_result,
    output_match,
    speedup
):

    return {

        "success":
            python_result["success"]
            and binary_result["success"]
            and output_match,

        "output_match": output_match,

        "python_runtime": python_result["runtime"],

        "compiled_runtime": binary_result["runtime"],

        "speedup": speedup,

        "python_output": python_result["stdout"],

        "compiled_output": binary_result["stdout"],

        "python_error": python_result["stderr"],

        "compiled_error": binary_result["stderr"]

    }


def print_validation_report(report):

    print("=" * 80)
    print("VALIDATOR AGENT")
    print("=" * 80)

    print(f"Validation Status : {'PASSED' if report['success'] else 'FAILED'}")

    print(f"Output Match      : {report['output_match']}")

    print(f"Python Runtime    : {report['python_runtime']:.6f} sec")

    print(f"Compiled Runtime  : {report['compiled_runtime']:.6f} sec")

    print(f"Speedup           : {report['speedup']:.2f}x")

    if not report["output_match"]:

        print("\nPython Output")
        print(report["python_output"])

        print("\nCompiled Output")
        print(report["compiled_output"])

    if report["python_error"]:

        print("\nPython Error")
        print(report["python_error"])

    if report["compiled_error"]:

        print("\nCompiled Error")
        print(report["compiled_error"])

    print("=" * 80)


def validator_agent(
    python_file,
    binary_path
):

    python_result = run_python(python_file)

    if not python_result["success"]:

        return create_validation_report(
            python_result,
            {
                "success": False,
                "stdout": "",
                "stderr": "",
                "runtime": 0
            },
            False,
            0
        )

    binary_result = run_binary(binary_path)

    if not binary_result["success"]:

        return create_validation_report(
            python_result,
            binary_result,
            False,
            0
        )

    output_match = compare_outputs(
        python_result,
        binary_result
    )

    speedup = calculate_speedup(
        python_result["runtime"],
        binary_result["runtime"]
    )

    report = create_validation_report(
        python_result,
        binary_result,
        output_match,
        speedup
    )

    return report

In [ ]:
def planner_wait():
    return """
    <div class="status-box">
    <div class="agent-title">🧠 Planner</div>
    🔄 Analyzing Python code...
    </div>
    """

def planner_done():
    return """
    <div class="status-box">
    <div class="agent-title">🧠 Planner</div>
    ✅ Completed
    </div>
    """

def builder_wait():
    return """
    <div class="status-box">
    <div class="agent-title">🔨 Builder</div>
    🔄 Generating & Compiling...
    </div>
    """

def builder_done():
    return """
    <div class="status-box">
    <div class="agent-title">🔨 Builder</div>
    ✅ Compilation Successful
    </div>
    """

def builder_failed():
    return """
    <div class="status-box">
    <div class="agent-title">🔨 Builder</div>
    ❌ Compilation Failed
    </div>
    """

def validator_wait():
    return """
    <div class="status-box">
    <div class="agent-title">✅ Validator</div>
    🔄 Running Validation...
    </div>
    """

def validator_done():
    return """
    <div class="status-box">
    <div class="agent-title">✅ Validator</div>
    ✅ Validation Passed
    </div>
    """

def validator_failed():
    return """
    <div class="status-box">
    <div class="agent-title">✅ Validator</div>
    ❌ Validation Failed
    </div>
    """

def create_benchmark_dataframe(report):

    return pd.DataFrame({

        "Metric": [

            "Python Runtime",

            "Compiled Runtime",

            "Speedup"

        ],

        "Value": [

            f"{report['python_runtime']:.6f} sec",

            f"{report['compiled_runtime']:.6f} sec",

            f"{report['speedup']:.2f}x"

        ]

    })

def build_execution_log(plan, build_result, validation):

    log = "# 📜 Execution Log\n\n"

    log += "## 🧠 Planner Agent\n"

    log += f"**Algorithm:** {plan['algorithm_type']}\n\n"

    log += f"**Time Complexity:** {plan['time_complexity']}\n\n"

    log += "Planner completed successfully.\n\n"

    log += "---\n\n"

    log += "## 🔨 Builder Agent\n"

    log += f"Compilation Attempts : {build_result['attempts']}\n\n"

    if build_result["success"]:

        log += "Compilation Successful\n\n"

    else:

        log += "Compilation Failed\n\n"

    log += "---\n\n"

    log += "## ✅ Validator Agent\n"

    if validation["success"]:

        log += "Output Match : True\n\n"

    else:

        log += "Output Match : False\n\n"

    log += f"Speedup : {validation['speedup']:.2f}x\n"

    return log

def get_compile_config(target_language):

    is_windows = platform.system() == "Windows"

    if target_language == "Rust":

        extension = "rs"

        binary = "main.exe" if is_windows else "./main"

        compile_command = [
            "rustc",
            "main.rs",
            "-O",
            "-o",
            binary
        ]

    else:

        extension = "cpp"

        binary = "main.exe" if is_windows else "./main"

        compile_command = [
            "g++",
            "main.cpp",
            "-O3",
            "-std=c++17",
            "-o",
            binary
        ]

    return compile_command, extension, binary

def optimize_pipeline(
    python_code,
    target_language,
    model_name,
    retries
):
    yield (

        planner_wait(),

        """
        <div class="status-box">
        <div class="agent-title">🔨 Builder</div>
        ⏸️ Waiting...
        </div>
        """,

        """
        <div class="status-box">
        <div class="agent-title">✅ Validator</div>
        ⏸️ Waiting...
        </div>
        """,

        "# 🚀 Pipeline Started",

        {},

        "",

        benchmark_df,

        {}

    )

    try:

        plan = planner_agent(
            client,
            model_name,
            python_code
        )

        if target_language:
            plan["recommended_language"] = target_language

        yield (

            planner_done(),

            builder_wait(),

            """
            <div class="status-box">
            <div class="agent-title">✅ Validator</div>
            ⏸️ Waiting...
            </div>
            """,

            "# 🧠 Planner Finished\n\nOptimization plan generated.",

            plan,

            "",

            benchmark_df,

            {}

        )

        compile_cmd, extension, binary = get_compile_config(target_language)

        build = builder_agent(

            client=client,

            model=model_name,

            python_code=python_code,

            plan=plan,

            compile_command=compile_cmd,

            extension=extension,

            max_retries=retries

        )

        if not build["success"]:

            yield (

                planner_done(),

                builder_failed(),

                validator_failed(),

                "# ❌ Compilation Failed",

                plan,

                build["source_code"],

                benchmark_df,

                {
                    "compile_error": build["compile_error"]
                }

            )

            return

        yield (

            planner_done(),

            builder_done(),

            validator_wait(),

            "# 🔨 Builder Finished\n\nCompilation Successful.",

            plan,

            build["source_code"],

            benchmark_df,

            {}

        )

        with open("program.py", "w") as f:

            f.write(python_code)

        validation = validator_agent(
            "program.py",
            binary
        )

        benchmark = create_benchmark_dataframe(validation)

        execution_log = build_execution_log(

            plan,

            build,

            validation

        )

        if validation["success"]:

            validator_box = validator_done()

        else:

            validator_box = validator_failed()

        yield (

            planner_done(),

            builder_done(),

            validator_box,

            execution_log,

            plan,

            build["source_code"],

            benchmark,

            validation

        )

    except Exception as e:

        traceback.print_exc()

        yield (

            planner_done(),

            builder_failed(),

            validator_failed(),

            f"# ❌ Error\n\n```\n{str(e)}\n```",

            {},

            "",

            benchmark_df,

            {
                "error": str(e)
            }

        )

In [ ]:
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="gray"
)

CUSTOM_CSS = """
footer {
    display:none;
}

.gradio-container{
    max-width:1700px !important;
}

.status-box{
    border-radius:12px;
    padding:12px;
    margin-bottom:12px;
    background:#f8f9fa;
    border:1px solid #dcdcdc;
}

.agent-title{
    font-size:18px;
    font-weight:bold;
}

.metric{
    font-size:16px;
}
"""

EMPTY_LOG = """
# Execution Log

Waiting for execution...
"""

EMPTY_PLAN = """
Planner output will appear here.
"""

EMPTY_CODE = ""

EMPTY_REPORT = """
Validation report will appear here.
"""

benchmark_df = pd.DataFrame(
    {
        "Metric": [
            "Python Runtime",
            "Compiled Runtime",
            "Speedup"
        ],
        "Value": [
            "-",
            "-",
            "-"
        ]
    }
)

with gr.Blocks(
    theme=theme,
    css=CUSTOM_CSS,
    title="AI Code Optimization Agent"
) as demo:

    gr.Markdown(
        """
# 🚀 AI Code Optimization Agent

Optimize Python code using an autonomous multi-agent pipeline.

**Planner → Builder → Validator**
"""
    )

    with gr.Row():

        with gr.Column(scale=3):

            python_code = gr.Code(
                label="Python Source Code",
                language="python",
                lines=24,
                value=""
            )

            with gr.Row():

                target_language = gr.Radio(
                    ["Rust", "C++"],
                    value="Rust",
                    label="Target Language"
                )

                model_name = gr.Dropdown(
                    choices=[
                        "gpt-5",
                        "gpt-5-mini",
                        "gpt-5-nano",
                        "gpt-4.1"
                    ],
                    value="gpt-5",
                    label="Model"
                )

                retries = gr.Slider(
                    minimum=1,
                    maximum=5,
                    value=3,
                    step=1,
                    label="Max Retries"
                )

            optimize_btn = gr.Button(
                "🚀 Optimize Code",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=1):

            gr.Markdown("## 🤖 Agent Status")

            planner_status = gr.Markdown(
                """
                <div class="status-box">
                <div class="agent-title">🧠 Planner</div>
                ⏸️ Waiting...
                </div>
                """
            )

            builder_status = gr.Markdown(
                """
                <div class="status-box">
                <div class="agent-title">🔨 Builder</div>
                ⏸️ Waiting...
                </div>
                """
            )

            validator_status = gr.Markdown(
                """
                <div class="status-box">
                <div class="agent-title">✅ Validator</div>
                ⏸️ Waiting...
                </div>
                """
            )

    with gr.Tabs():

        with gr.Tab("📜 Execution Log"):

            execution_log = gr.Markdown(
                value=EMPTY_LOG
            )

        with gr.Tab("🧠 Optimization Plan"):

            planner_output = gr.JSON(
                value={}
            )

        with gr.Tab("💻 Generated Code"):

            generated_code = gr.Code(
                language="cpp",
                label="Generated Code",
                lines=24,
                value=EMPTY_CODE
            )

        with gr.Tab("📊 Benchmark"):

            benchmark_table = gr.Dataframe(
                value=benchmark_df,
                interactive=False,
                wrap=True
            )

        with gr.Tab("✅ Validation Report"):

            validation_report = gr.JSON(
                value={}
            )
            
        optimize_btn.click(

            fn=optimize_pipeline,

            inputs=[

                python_code,

                target_language,

                model_name,

                retries

            ],

            outputs=[

                planner_status,

                builder_status,

                validator_status,

                execution_log,

                planner_output,

                generated_code,

                benchmark_table,

                validation_report

            ]

        )

demo.queue()
demo.launch()